In [1]:
import nibabel as nib
from pathlib  import Path
import numpy as np
from tqdm import tqdm
import os

In [2]:
RESULT_FOLDER = Path('../results')
SURF_FILE = "/home/guoqiu/guoqiu/Database/HCP_S1200_GroupAvg_v1/S1200.L.midthickness_MSMAll.32k_fs_LR.surf.gii"

In [3]:
# nii -> nii_by_label
for img_file in (RESULT_FOLDER / 'nii').iterdir():
    img = nib.load(img_file)
    
    img_data = np.round(img.get_fdata()).astype(int)
    max_n = img_data.max()
    for i in range(1, max_n + 1):
        new_img_data = (img_data == i).astype(int)
        new_img = nib.Nifti1Image(new_img_data, affine=img.affine, dtype=np.int8)
        new_img_file = RESULT_FOLDER / 'nii_by_label' / f'{img_file.name[:-7]}_label{i}.nii.gz'
        nib.save(new_img, new_img_file)

In [4]:
# nii_by_label -> gii_by_label
from_file_list = list((RESULT_FOLDER / 'nii_by_label').iterdir())
for from_file in tqdm(from_file_list):
    from_file_str = str(from_file.resolve())
    to_file_str = (from_file_str
                   .replace('/nii_by_label/', '/gii_by_label/')
                   .replace('.nii.gz', '.func.gii'))
    shell_code  = f'wb_command -volume-to-surface-mapping "{from_file_str}" "{SURF_FILE}" "{to_file_str}" -enclosing'
    os.system(shell_code)

100%|█████████████████████████████████████████| 320/320 [01:00<00:00,  5.31it/s]


In [5]:
# nii -> gii
from_file_list = list((RESULT_FOLDER / 'nii').iterdir())
for from_file in tqdm(from_file_list):
    from_file_str = str(from_file.resolve())
    to_file_str = (from_file_str
                   .replace('/nii/', '/gii/')
                   .replace('.nii.gz', '.func.gii'))
    shell_code  = f'wb_command -volume-to-surface-mapping "{from_file_str}" "{SURF_FILE}" "{to_file_str}" -enclosing'
    os.system(shell_code)

100%|███████████████████████████████████████████| 25/25 [00:04<00:00,  5.28it/s]
